## 04 - TME Deconvolution Plan (XDec + CIBERSORTx + TIMER)

### Aim
* Plan a tumor microenvironment (TME) deconvolution pass on the QC'd TCGA-PAAD matrix, comparing three tools: XDec (reference-optional cancer-cell/stromal state discovery), CIBERSORTx (reference-based immune cell fraction estimation, LM22), and TIMER (reference-based, cancer-type-specific immune fraction estimation). **Planning only - no code is executed in this notebook.**

### Input
* `data/processed/TCGA-PAAD_counts_raw.tsv` (notebook 02) - raw integer gene-level counts, `gene_symbol` row index - now the primary input matrix, since XDec's own pipeline expects raw counts and does its own internal normalization (see below). `TCGA-PAAD_TPM_raw.tsv` is kept as a derived input for the tools (CIBERSORTx, TIMER) whose own documentation recommends TPM/FPKM over raw counts.
* `TCGA-PAAD_sample_metadata.tsv` (notebook 02).
* `data/qc/flagged_samples.tsv`, `data/qc/batch_effect_check.txt` (notebook 03) - the open sample-inclusion and batch-effect questions this deconvolution run is meant to help resolve.

### Output
* This planning document: corrected tool introductions (verified directly against each tool's GitHub repo rather than assumed), a gene-ID/GEO-matching strategy, an R-vs-Python recommendation, and a numbered implementation plan. No data files are produced here.

### Steps
* See "Planned pipeline" section below for the full numbered sequence.

### Result
* N/A - this is a planning document, nothing has been run yet.

### Conclusion
* XDec (`BRL-BCM/XDec_SM`) is confirmed **Python**, not R as previously assumed in this notebook - it explicitly wants raw integer counts and does its own quantile/logistic normalization internally. CIBERSORTx remains Docker/token-gated regardless of language - no official Python (or R) package exists for the core SVR engine itself. TIMER adds a third, cancer-type-specific immune reference; its original repo is archived in favor of the maintained `immunedeconv` R wrapper. Plan is to feed all three tools from the single `TCGA-PAAD_counts_raw.tsv` matrix (each tool applying whatever normalization it specifically documents needing), standardize on HGNC gene symbols as the shared ID namespace, and resolve any external reference's gene IDs via its GEO platform (GPL) annotation before intersecting gene sets. Ready to implement as follow-up notebooks (`05_xdec_deconvolution.ipynb` / `06_cibersortx_deconvolution.ipynb` / `07_timer_deconvolution.ipynb`) once the open decisions at the end of this notebook are resolved.

## Why three tools, not one

XDec, CIBERSORTx, and TIMER answer overlapping-but-distinct questions and are complementary rather than redundant:

* **XDec** is *reference-optional* and data-driven. It doesn't require a pre-specified catalog of cell types - it discovers recurrent transcriptional **states** directly from the bulk tumor profiles themselves, including novel cancer-cell states that don't correspond to any single-cell reference. This is the right tool for the cancer-cell and stromal-state side of the question (e.g. Basal-like vs. Classical PDAC cancer-cell states; myCAF vs. iCAF vs. necrotic/hypoxic CAF subtypes building the desmoplastic reaction - **not** a CAF-vs-adipocyte contrast, which was an incorrect carryover from breast cancer stroma and doesn't apply to pancreas) - exactly what the XDec-SM and XDec-CHI papers in this project's `papers/` folder were built to do.
* **CIBERSORTx** is *reference-based*. It requires a signature matrix of known, purified cell-type expression profiles (e.g. the 22-immune-cell-type LM22 matrix) and fits bulk mixtures as combinations of those fixed references. This is the right tool for well-characterized **immune cell** composition, where decades of sorted-reference data give it strong validation and fine-grained immune subtyping (naive vs. memory T cells, M1 vs. M2 macrophages, etc.) that a reference-optional method like XDec isn't designed to resolve on its own.
* **TIMER** is also *reference-based*, but with a **cancer-type-specific** immune reference (built per tumor type, including PAAD) rather than CIBERSORTx's blood-derived LM22 - it's the right tool for checking whether immune fractions look different when the reference is matched to the tumor type being deconvolved, rather than to sorted peripheral blood. It resolves fewer immune subsets than LM22 (6 broad TIIC types vs. LM22's 22) but is the one method in this comparison built and validated specifically on TCGA-style tumor data.

Running all three and cross-checking where they overlap (stromal/immune proportions) mirrors exactly what the XDec-SM paper itself did as a validation step (Fig 2/3: XDec-SM vs. CIBERSORTx, EPIC, xCell, Kassandra, InfiniumPurify, **TIMER**) - TIMER was already one of the paper's own comparison methods, so adding it here closes the gap with the published validation set rather than introducing a new axis of comparison.

**PDAC TME context (corrects the CAF-vs-adipocyte framing above):** PDAC's microenvironment is structurally dominated by an immunosuppressive myeloid network (MDSCs, TAMs, tumor-associated neutrophils, Tregs) embedded in a dense hyaluronan-rich stroma that physically excludes immune cells. PDAC CAFs stratify into myofibroblastic (myCAF), inflammatory (iCAF), and necrotic/hypoxic subtypes with distinct roles in building that desmoplastic reaction. Composition and physical architecture work together to seal the tumor off - adipocytes are not a feature of this biology. This reframes what XDec's stromal-state discovery should actually be targeting, and what any reference (e.g. `GSE111672`, see below) needs to resolve: CAF-subtype heterogeneity, not a CAF/adipocyte axis.

## XDec: package and algorithm

**What it is:** XDec-SM / XDec-CHI (Murillo et al. 2023, *PLOS Comp Bio*; Lewis/LaPlante et al. 2022, *iScience* - both in `papers/`) is an extension of **EDec** (the authors' earlier DNA-methylation deconvolution method), adapted for bulk RNA-seq.

**Package (corrected):** `https://github.com/BRL-BCM/XDec_SM`, from the Milosavljevic lab (Baylor College of Medicine) - the same group behind `EDec`. **This repo is Python, not R** (verified directly against the repo contents; an earlier version of this notebook incorrectly stated "R only, no Python implementation exists"). Install via `pip install numpy pandas scipy quadprog` (numpy>=1.24, pandas>=2.0, scipy>=1.11, quadprog>=0.1.11 recommended; see `requirements-optional.txt` for optional Census/container adapters). Repo layout: `python/xdec.py` (the three-stage CLI tool), `python/preprocess_scrna.py` (aggregates scRNA-seq into pseudobulk references), `python/xdec_auto.py` (automated stage0→stage1→stage2 wrapper), `python/gui.py` (Streamlit workbench), `tools/geoquery_inventory.R` (fetches GEO supplementary files/sample metadata by accession for building references - see "Gene ID / GEO matching" below), `tools/repair_numeric_gene_ids.py` (relabels CELLxGENE Census references that only carry numeric feature indices). `XDec_CHI` (the PDAC-specific sliding-window variant) is a separate repo - not yet re-checked, so its R-vs-Python status should be re-verified before relying on the "R" claim for that one too.

**Algorithm (3 stages, as implemented in `python/xdec.py`):**
1. **Stage 0 (optional):** identify an informative gene set via Welch's t-test one-vs-rest selection on a reference (scRNA-seq pseudobulks, sorted cell lines, or a literature gene signature like PAM50). References are used *only* to pick which genes matter, never for quantitative values - this is what makes the method "reference-optional." CLI: `--bulk` (required), `--refs`, `--ref-metadata`, `--probes` (skip selection if pre-computed), `--k-range`, `--p-one-vs-rest`, `--n-one-vs-rest`.
2. **Stage 1:** constrained matrix factorization via quadratic programming (`V ≈ W x H`). Confirmed normalization sequence: quantile-normalize the sample×gene matrix, then apply the paper's logistic rescaling per gene (`y = 1 - e^{-ax}`, `a = 1/max(x)`), then divide the whole matrix by its global max, then drop genes with mean rescaled expression < 0.01. Stage-1 convergence is checked via **RSS** (residual sum of squares, `abs(rss_prev - rss) < 1e-10`) across iterations, not RMSE - worth flagging since RMSE is sometimes assumed to be the metric here. Number of states (k) chosen by a stability procedure (3 random-subsample replicates across a k-range, default up to ~10/12), writing `stability_scores.tsv` and `optimal_k.txt`; the exact stability-score formula isn't documented in the README and would need to be read from source if it matters for reporting. CLI: `--bulk` (required), `--probes` or `--use-all-genes`, `--k` (reads `optimal_k.txt` if omitted), `--anchor-refs`.
3. **Stage 2:** non-negative least squares (also via quadratic programming) using the Stage-1 proportions against the **raw, untransformed count matrix** (`--bulk-counts`, required) plus `proportions.tsv` from Stage 1, to recover cell-type-specific expression profiles on the original count scale; reports per-profile standard errors from residual variance (`σ² = ‖residuals‖² / (n_samples − k)`).

**Raw-count input requirement (new finding, relevant to this plan's switch to `TCGA-PAAD_counts_raw.tsv`):** the pipeline's `fetch-counts` step explicitly **rejects** filenames/content indicating pre-normalized data (TPM, FPKM, RPKM, CPM, "normalized", "scaled", VST, rlog, log-CPM) and validates that matrices are integer, non-negative raw counts; `--log2-bulk`/`--log2-refs` flags exist only to *invert* a prior log2(x+1) transform if someone's input was already logged. In other words, XDec wants exactly the kind of matrix `TCGA-PAAD_counts_raw.tsv` already is, and does its own quantile/logistic/max-scale normalization internally rather than expecting a pre-normalized TPM matrix - the original plan's choice of feeding it `TCGA-PAAD_TPM_raw.tsv` was not aligned with how the tool is actually built to be used.

**Gene ID format:** no single mandated namespace is enforced in code - the first column is read as a string gene/probe identifier and matched by exact string equality across bulk/reference/probe-list inputs. In practice the repo's own examples and hardcoded gene lists (e.g. `PAM50_GENES`) use HGNC gene symbols, and CELLxGENE Census-derived references are explicitly resolved to `feature_name` gene symbols (summing duplicates at the raw-count level) rather than left as numeric indices - numeric-only gene IDs are rejected for that reference path. `TCGA-PAAD_counts_raw.tsv`'s `gene_symbol` index is directly compatible.

**XDec-CHI adds:** a sliding-window deconvolution across samples ranked by a gene of interest, repeating Stage 2 on overlapping windows of ≥40 samples to get cell-type-*specific* gene-gene correlations - this is how the PDAC paper found cancer-cell `MSLN` correlating with stromal `IFNGR1`, for example. This is the method for discovering intracellular/heterotypic interactions, not just getting a one-shot proportions table.

## CIBERSORTx: package and algorithm

**What it is:** CIBERSORTx (Newman et al. 2019, *Nat Biotechnol*) is the successor to the original CIBERSORT, from the Alizadeh/Newman lab (Stanford).

**Algorithm:** support vector regression (nu-SVR) - bulk expression for each sample is modeled as a linear mixture of reference cell-type profiles, and nu-SVR fits the mixing proportions while being robust to noise and to the presence of cell types not in the reference (unlike ordinary least squares). Three modes relevant here:
* **Fractions mode:** per-sample relative (or, with normalization, absolute) proportions of each reference cell type.
* **High-Resolution (Hi-Res) mode:** cell-type-specific gene expression profiles per sample - the CIBERSORTx analogue of XDec's Stage 2 output.
* **Create Signature Matrix:** builds a custom reference signature matrix from scRNA-seq or sorted-bulk input, instead of using the default.

**Default reference:** **LM22**, a 22-immune-cell-type signature matrix built from sorted leukocyte expression profiles (not tumor-derived) - this is the standard reference most published CIBERSORTx TME studies use, including the XDec-SM comparison.

**Package / access model - re-checked for a Python package:** CIBERSORTx's core SVR engine is still **not** distributed as an installable CRAN or PyPI package by Stanford - it remains gated behind a Docker container (`cibersortx/fractions`, `cibersortx/hires`) run against a registration token from `https://cibersortx.stanford.edu`, or the equivalent web interface. What does exist in the Python ecosystem, and should not be confused with an official CIBERSORTx package: (1) `jason-weirather/CIBERSORT` on GitHub, a Python CLI/module wrapper - but it drives the **original 2015 CIBERSORT** algorithm, which still requires separately obtaining the licensed source from the Newman/Alizadeh lab, not CIBERSORTx's gated SVR model; (2) assorted community forks with small Python helper scripts (e.g. `convert_reference_file.py`-style utilities) for preparing scRNA-seq reference files to feed into the Docker image - I/O plumbing, not the engine itself. Bottom line: "Python" doesn't remove the Docker/token requirement for CIBERSORTx proper, the same way "R" didn't before - any Python package found is either the unrelated original CIBERSORT or a pre/post-processing helper around the same Docker image.

**Input format caveat (relevant to this plan's raw-count matrix):** CIBERSORTx's own guidance calls for gene-symbol-indexed, non-log **linear** mixture values, with TPM/FPKM/microarray intensity recommended; raw counts aren't the documented input because they don't correct for library-size differences across samples the way TPM does. For this plan, CIBERSORTx should be run on a TPM matrix derived from `TCGA-PAAD_counts_raw.tsv` (i.e. reuse `TCGA-PAAD_TPM_raw.tsv` from notebook 02) rather than the raw counts directly - XDec is the one tool in this comparison actually designed to take raw counts as its primary input.

## TIMER: package and algorithm

**What it is:** TIMER (Li et al. 2016 *Genome Biology*; TIMER2.0 web server, Li et al. 2020 *NAR*) estimates the abundance of six tumor-infiltrating immune cell (TIIC) types - B cells, CD4 T cells, CD8 T cells, neutrophils, macrophages, dendritic cells - using a constrained-regression deconvolution against immune reference profiles that are **built per cancer type** (including a PAAD-specific reference), rather than a single generic signature like LM22.

**Repo named in this plan's update:** `https://github.com/hanfeisun/TIMER`. Checked directly: this is a Python/R mixed repo (Conda `environment.yml`, `make test`) that packages the **original** TIMER source from `cistrome.org/TIMER/download.html`. The repo is explicitly **archived/deprecated by its own maintainer**, with a note directing users to the **VIPER** RNA-seq pipeline, which has absorbed TIMER as one of its immune-infiltration modules. It also depends on an external Dropbox link for reference expression data and cancer-specific marker genes - an unversioned, fragile dependency for a reproducible pipeline.

**Maintained alternative:** the `timer` method inside the R package `immunedeconv` (already noted as an optional package in notebook `00_install_packages.ipynb`) wraps the same underlying algorithm with a maintained, versioned implementation, no Dropbox dependency, and the same one-call interface used for the other `immunedeconv` methods already mentioned in this plan (xCell, EPIC, quanTIseq, MCPcounter, ESTIMATE).

**Input format:** gene × sample matrix, **TPM/RPKM/FPKM, not log-transformed, not raw counts** - same caveat as CIBERSORTx, so TIMER should also run on the TPM-derived matrix, not `TCGA-PAAD_counts_raw.tsv` directly. Rows are expected as **HGNC gene symbols**; `immunedeconv::deconvolute(method = "timer")` additionally requires an `indications` vector giving each sample's cancer-type code (`"paad"` here), since the reference itself is cancer-type-specific - this is a required argument, not optional metadata.

## Gene ID / GEO matching

All three tools above turn out to key on **HGNC gene symbols** in practice (XDec by convention/example, not enforcement; CIBERSORTx's LM22 by construction; TIMER/`immunedeconv` by requirement), and `TCGA-PAAD_counts_raw.tsv`'s `gene_symbol` row index already matches that namespace - so no ID conversion is needed for the TCGA-PAAD matrix itself.

The place ID matching actually becomes a problem is any **external reference** pulled in for XDec Stage 0 (a PDAC scRNA-seq informative-gene reference) or for a custom CIBERSORTx/TIMER signature matrix. XDec's own `tools/geoquery_inventory.R` fetches a dataset's supplementary files and sample metadata by **GEO accession** (`GEOquery::getGEOSuppFiles()` / `getGEO()`), but - checked directly - it only inventories files and phenotype data, it does **not** perform any gene-ID mapping itself. Plan: if a GEO-hosted reference is used, (1) fetch it by its **GSE** accession the same way `geoquery_inventory.R` does, (2) pull the dataset's **GPL** platform annotation table (`GEOquery::getGEO("GPLxxxx")`'s feature data) to map that platform's probe/gene IDs to HGNC symbols, (3) collapse any duplicate symbols (sum, matching how XDec's Census-reference repair script handles duplicate `feature_name`s), then (4) intersect with the TCGA-PAAD symbol set before running Stage 0 selection or building a custom signature matrix. Don't assume a downloaded reference's row labels already line up with TCGA gene symbols without this step.

One caveat: the XDec-CHI paper's own PDAC scRNA-seq reference (Peng et al. 2019) was deposited on **GSA** (China National GeneBank, accession `CRA001160`), not GEO - `GEOquery` won't fetch it. If Stage 0 option (a) below (re-deriving the informative gene set from scRNA-seq) is chosen and a GEO-based fetch workflow is wanted end-to-end, a GEO-hosted PDAC scRNA-seq/signature dataset would need to be identified as a substitute or supplement, rather than assuming Peng et al. is reachable via this route.

## R or Python?

**Revised recommendation: XDec is a deliberate, scoped Python exception; CIBERSORTx and TIMER pre/post-processing stay in R.**

* **XDec is Python-only** (corrected above) - the opposite of what this notebook previously assumed. This directly conflicts with this project's house convention (`CLAUDE.md`: all analysis as `ir`-kernel R notebooks). Two ways to resolve that, both legitimate, flagged as an open decision below rather than decided here: (a) a dedicated Python-kernel notebook for the XDec steps only, documented as an explicit exception; or (b) keep an `ir`-kernel notebook and shell out to the `xdec.py` CLI via `system2()`, so the notebook's kernel stays R while the actual Stage 0-2 computation runs as a subprocess and results are read back into R for everything downstream (cross-validation, plotting, QC resolution).
* **CIBERSORTx is still language-agnostic infrastructure** (Docker CLI, token-gated) regardless of this notebook's earlier claim about a Python package - re-checked above, no real Python package exists for the engine itself. Pre/post-processing (building the TPM input, reading proportions back) stays in R, same reasoning as before: one environment, same TPM matrix and gene-symbol conventions already used in notebooks 00-03.
* **TIMER's maintained path is R** - `immunedeconv::deconvolute(method = "timer")` runs natively in the same `pixi` R environment, vs. the archived `hanfeisun/TIMER` repo's Conda/Dropbox setup. No cross-language step needed here unless the archived repo is deliberately chosen for exact reproduction of the original method.
* If CIBERSORTx's Docker/token registration is undesirable, the registration-free R alternatives remain `immunedeconv` (also the TIMER wrapper, so one package covers both) or `IOBR`, both already noted as optional in `00_install_packages.ipynb`.

## Comparing candidate scRNA-seq references for XDec Stage 0

Every PDAC single-cell/reference source checked, GEO-restricted and not. "Verified" = checked directly (file listing, download, or object inspection), not taken from a secondary summary.

| Reference | Resource | Cell type coverage | Sample size | Note |
|---|---|---|---|---|
| `GSE111672` (Moncada 2020) | GEO | General (ductal/cancer/stromal/immune); one generic `Fibroblasts` label, no subtype split | 2 patients, 3,659 cells | **Verified on download**: only 5 fibroblast cells in PDAC-A, 0 in PDAC-B - no CAF-subtype resolution possible |
| `GSE155698` (Steele 2020) | GEO | Immune-focused (T/myeloid/NKT); epithelial/stromal secondary | 17 tumor + 3 normal | Risks Stage-0's t-test being dominated by immune-vs-rest contrasts; sorting status not independently confirmed |
| `GSE205049`/`GSE205354` (Hwang 2023) | GEO | Immune-dysfunction focus; scRNA+spatial+CyTOF | Treatment-naive PDAC + normal + blood | Sorting markers unverified (paywalled primary source) |
| `GSE154778` | GEO | Whole dissociated tissue - not FACS-sorted, no ready labels, full coverage by construction | 10 primary + 6 metastatic (16 patients) | Would need its own clustering/annotation sub-pipeline first - self-derived labels, same caveat as proxy clusters |
| `GSE129455` (Elyada 2019) | GEO | Fibroblast-enriched (sorted CD45−/EPCAM−/CD31−); defines myCAF/iCAF/apCAF | 4 tumors | **Mouse only** - human data from the same study is dbGaP, not GEO |
| `GSE212966`, `GSE263733` | GEO | Unverified | Unverified | Title-only hits, not independently checked |
| Loveless et al. 2024 (`Zenodo 14199536`) | Zenodo | Named **CXCL10+ fibroblast** subtype; broad atlas (~21 cell types); validated directly against TCGA-PAAD bulk | 229 patients, ~1M cells | **Verified via Zenodo API**: single 33.4 GB file - exceeds 16GB RAM, infeasible |
| `researchdata.se/2025-64` | SND | Named **myCAF/iCAF/ifCAF** | 5 patients | **Restricted access** - requires a formal data-access request |
| HTAN | Synapse | Large, rigorously curated, multi-consortium | 200-257 patients, 1.6M+ cells | Synapse account + DUA required; even larger than Loveless et al. - same RAM infeasibility |
| CELLxGENE Census | Census | Unknown | - | XDec has native tooling for this (`census_adapter.py`) but no PDAC CAF-annotated entry confirmed |
| `CAF_integrated.rds`, considered then superseded | Zenodo `6024273` (integrates `GSE111672`+`GSE154778`+`GSE155698`+`GSM4293555`+Peng et al.+OUGS) | **CAF/fibroblast-only** - all 10,504 cells are fibroblasts; 7 unlabeled `seurat_clusters` used as a proxy for subtype | 10,504 cells | **Ran successfully** (262 real genes, no errors) - but the one-vs-rest test it produced is CAF-internal only ("cluster X vs other CAF clusters"), not a whole-tumor contrast. Superseded once a full-coverage PDAC-specific option was confirmed accessible. |
| Peng et al. 2019 / `CRA001160`, raw | GSA | ~10 cell types, PDAC-specific | 35 patients (24 PDAC + 11 normal) | **Verified on the GSA entry**: 70 files, 2.69 TB raw sequencing runs only - infeasible to reprocess from scratch |
| **Peng et al. 2019, reprocessed - chosen** | **Zenodo `3969339`** ("Peng2019 PRJCA001063 ... for Besca") | **Full coverage, PDAC-specific, better stromal split than any GEO-only option**: ductal (21,487), **fibroblast (6,926) + pancreatic stellate cell (5,862) as two separate stromal classes**, endothelial (9,027), acinar, endocrine, detailed immune (macrophage, B cell, CD4/CD8 T-cell subsets incl. regulatory/proliferating, myeloid DC, plasma cells) | **57,423 cells**, 35 patients - the real, (near-)complete Peng et al. dataset, reprocessed, not a subset | **Downloaded and verified** (`StdWf1_PRJCA001063_annotated.h5ad`, 1,681,192,568 bytes, exact match); clean HGNC gene symbols. **Technical wrinkle** (see note below) |

**Technical wrinkle found on inspection of `Zenodo 3969339` - not a blocker, but must be handled correctly:** neither matrix in the annotated h5ad is literal raw integer counts.
* `adata.X` (the main matrix) is z-scored/scaled - contains **negative values** (range -4.18 to 10.0) - not usable directly as `--refs` (XDec expects non-negative expression).
* `adata.raw.X` is non-negative (0 to 8.93) but is **natural-log-normalized** (`ln(x+1)` via `sc.pp.log1p()`), not `log2(x+1)`. XDec's `--log2-refs` flag inverts `log2(x+1)` specifically (`2^x - 1`) - applying it here would use the wrong base and produce incorrect values.
* **Fix:** back-transform `adata.raw.X` with `expm1()` (= `e^x - 1`, the correct inverse for natural log) myself in Python before writing the `--refs` TSV, then feed XDec the already-linearized values **without** passing `--log2-refs` (since the data is no longer log-transformed by that point - inverting a non-log2 transform via XDec's own flag would double-count/mis-transform it).
* Separately noticed: this file's cell barcodes (e.g. `T1_AAACCTGAGATGTCGG`) match the barcode format seen in `CAF_integrated.rds`'s `Patient2` column (`CA001063_T1`) - confirming Peng et al.'s cells are indeed one of the five source datasets integrated into `CAF_integrated.rds`, i.e. the double-counting risk flagged earlier when considering combining `GSE111672` + `CAF_integrated.rds` was a real, verifiable concern, not just a theoretical one.

**Why `Zenodo 3969339` (Peng et al., reprocessed) wins over everything else:** it is the only option that is simultaneously (a) full whole-tumor cell-type coverage, (b) PDAC-specific rather than a different/smaller pancreatic study, (c) large (57K cells, 35 patients), (d) the *exact* source data behind the PMID 36274954 benchmark paper, and (e) small enough to load on this laptop (1.68 GB). Every other option failed at least one of these - `GSE111672` (small, thin CAF depth), `CAF_integrated.rds` (CAF-only, no whole-tumor contrast), Loveless et al./HTAN (too large), `researchdata.se`/dbGaP (access-gated). The one real cost is the log-transform handling above, which is a data-wrangling detail, not a feasibility or coverage problem.

## Planned pipeline

1. **Resolve sample inclusion first.** The 8 PCA-outlier and 4 sex-mismatch samples from notebook 03 are still an open question. Plan: run deconvolution on the *full* 182-sample matrix first (don't pre-exclude), then use the resulting proportions to answer the question notebook 03 raised - do the 8 PCA outliers' proportions track `tumor_grade` (keep them) or `TSS` with no biological story (exclude / batch-correct)? This also resolves the step-8 batch-effect question from the same notebook. Decide sample exclusion *after* this check, not before.

2. **Settle the gene-ID namespace.** Confirm `TCGA-PAAD_counts_raw.tsv`'s `gene_symbol` index is used as-is (HGNC symbols) for all three tools. `Zenodo 3969339`'s gene symbols are already clean HGNC - no remapping needed there either.

3. **XDec Stage 0 - informative gene set. Decided (supersedes both `GSE111672` and `CAF_integrated.rds`): `Zenodo 3969339`** - Peng et al. 2019's actual PDAC scRNA-seq data (`PRJCA001063`/`CRA001160`), reprocessed and annotated by Roche's Besca pipeline. Chosen because it is simultaneously full whole-tumor coverage, PDAC-specific, large (57,423 cells, 35 patients), the exact source behind the PMID 36274954 benchmark paper, and small enough for this laptop (1.68 GB) - see the comparison table above for why every alternative failed at least one of these.
   * **Downloaded and verified:** `progress/data/raw/zenodo_3969339/StdWf1_PRJCA001063_annotated.h5ad`. 57,423 cells, clean HGNC gene symbols. Cell types: ductal (21,487), fibroblast (6,926) + pancreatic stellate cell (5,862) as two separate stromal classes, blood vessel endothelial (9,027), pancreatic acinar (2,048), plus detailed immune (macrophage, B cell, CD4/CD8 T-cell subsets incl. regulatory/proliferating, myeloid DC, plasma cells).
   * **Technical wrinkle to handle before use as `--refs`:** `adata.X` is z-scored (has negative values - unusable directly); `adata.raw.X` is non-negative but natural-log-normalized (`ln(x+1)`), not `log2(x+1)` - XDec's `--log2-refs` flag inverts the wrong base. Fix: back-transform `adata.raw.X` with `expm1()` in Python before writing the `--refs` TSV, then run XDec **without** `--log2-refs` (data is already linear by that point).
   * `GSE111672` and `CAF_integrated.rds` stay downloaded but are both superseded as the primary Stage-0 input - `GSE111672` for being smaller/thinner on CAF depth, `CAF_integrated.rds` for being CAF-only (its one-vs-rest test was CAF-internal, not whole-tumor). The CAF-internal 262-gene run and its partial k-sweep (k=3,4,5 completed, k=6 killed mid-run) stand as a side result, not the main Stage-0 gene set.

4. **XDec Stage 1.** Run `python/xdec.py stage1` directly on `TCGA-PAAD_counts_raw.tsv` (restricted to the informative genes selected in step 3) - the tool's own quantile-normalize → logistic-rescale (`1-e^{-ax}`) → max-scale → low-expression-filter sequence replaces any separate TPM pre-conversion for this step. k chosen from a (re-run) stability sweep against the real Peng-derived gene set, partitioned by k as before.

5. **XDec Stage 2.** Run `python/xdec.py stage2 --bulk-counts TCGA-PAAD_counts_raw.tsv --proportions <stage1 output>` to recover untransformed, per-cell-type expression profiles via NNLS. Expected to be fast (one-shot NNLS per gene, no stability-style repetition) - time it once rather than pre-emptively partitioning.

6. **(Optional) XDec-CHI sliding-window analysis.** Only needed if the goal extends beyond proportions to specific gene-gene interaction discovery (the PDAC paper's headline result, e.g. cancer-cell `MSLN` vs. stromal `IFNGR1`) - requires ≥40-sample windows, comfortably met by this 182-sample cohort. Re-verify XDec-CHI's own R-vs-Python status before implementing (not yet re-checked, unlike XDec-SM above).

7. **CIBERSORTx Fractions mode.** Derive (or reuse) a TPM matrix from the same raw counts (`TCGA-PAAD_TPM_raw.tsv` from notebook 02) - CIBERSORTx's documented input is non-log linear TPM/FPKM, not raw counts (see caveat above). Register for a token, confirm Docker is available locally, run `cibersortx/fractions` (LM22 default signature - no reference-sourcing needed, unlike XDec) against that TPM matrix.

8. **TIMER.** Decide archived `hanfeisun/TIMER` (exact original method, Conda + Dropbox dependency) vs. `immunedeconv::deconvolute(method = "timer")` (recommended, maintained, same R environment) - see open decisions below. Feed the same TPM-derived matrix (HGNC symbols, not raw counts, not log-transformed) with each sample's cancer-type indication set to `"paad"`. No reference-sourcing needed (built-in cancer-type-specific reference).

9. **Cross-validate.** Compare XDec's summed stromal/immune proportions against CIBERSORTx's and TIMER's, pairwise and three-way - the same comparison the XDec-SM paper itself performed (its Fig 2/3 explicitly included TIMER alongside CIBERSORTx, EPIC, xCell, Kassandra) - a sanity check that all three broadly agree on the non-cancer compartment, even though XDec's cancer-cell-state resolution is the point neither CIBERSORTx nor TIMER can match.

10. **Resolve the open QC questions.** Use step 1's output to settle the PCA-outlier and batch-effect questions left open in notebook 03.

11. **Save outputs.** Proportions and cell-type-specific profiles from all three tools, written to a new `data/deconvolution/` directory alongside a short README documenting: the gene-ID namespace, the `zenodo_3969339` provenance and the log-transform handling, which input matrix (raw counts vs. TPM) and normalization each tool actually consumed, and which CIBERSORTx/TIMER reference was used - for reproducibility.

## Open decisions before implementing

* **Gene set for XDec Stage 0 - decided:** `Zenodo 3969339` (Peng et al. 2019, reprocessed) as `--refs`/`--ref-metadata`, full whole-tumor coverage with real curated cell-type labels. Supersedes both `GSE111672` (full coverage but thin CAF depth) and `CAF_integrated.rds` (CAF-only, no whole-tumor contrast). 344 real informative genes selected; full `k=3..12` stability sweep (partitioned by k) in progress.
* **XDec's Python-vs-house-R-convention conflict - decided:** `XDec` (notebook 06) and `CIBERSORTx` (notebook 07) both run on the **Python 3 kernel**, an explicit, documented exception to `CLAUDE.md`'s R-kernel convention for this project. XDec is Python-only, so running it natively avoids an R-shells-out-to-Python indirection layer; CIBERSORTx is language-agnostic (Docker), so Python here is a consistency choice rather than a requirement. `TIMER` (notebook 08) stays on the `ir` (R) kernel, since its actual algorithm (`hanfeisun/TIMER`'s `regression.r`/`utils.r`) is genuinely R code - no exception needed there.
* **TIMER source - decided:** the archived `hanfeisun/TIMER` repo directly (not `immunedeconv`, which turned out to be uninstallable via `pixi` - its dependency `r-xcell` has no bioconda build for R 4.5.3, only up to R 4.4). Ran successfully: found and fixed a real `sqldf`/`r-chron` dependency conflict by patching the local clone's `utils.r` (the function needing `sqldf` is never reached once `prec.zip`'s precalculated data is present). Full 182-sample cohort completed - see notebook 08.
* **CIBERSORTx access:** willing to register for a token and run Docker locally, or prefer to skip CIBERSORTx and use the registration-free R alternatives (`immunedeconv` / `IOBR`) for the reference-based comparison instead? Still open - notebook 07 is staged and ready, blocked only on the token.
* **Custom vs. default CIBERSORTx signature:** stick with LM22 (blood-derived, well-validated, used in the XDec-SM paper's own comparison) or build a PDAC-specific signature matrix from `Zenodo 3969339` (more tumor-relevant, but another processing step and another thing to validate)?
* **Sample set:** confirmed above to decide *after* step 1's post-hoc check, not before - flagged here as a reminder not to skip that check under time pressure.